# Notebook 14 — Synthetic Data Pipelines for LLM Training

    ## Learning objectives

    - Design generation, verification, filtering, deduplication, and curriculum stages
- Measure diversity, contamination, provenance, and teacher-induced bias
- Build a reproducible synthetic instruction-data pipeline with explicit quality gates

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['datasets>=3.5,<6', 'huggingface-hub>=0.30,<1']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 14.1 Synthetic data is a pipeline, not a prompt

Synthetic examples can expand coverage, translate formats, generate edge cases, distill a stronger
teacher, or create problems with mechanically verifiable answers. They do not create information for
free. Outputs inherit teacher capabilities, blind spots, policy, style, and correlations. Repeatedly
training models on unfiltered model outputs can narrow diversity and amplify errors.

A governed pipeline separates task specification, seed selection, generation, parsing, verification,
filtering, deduplication, balancing, split construction, human audit, versioning, and downstream
ablation. Preserve the raw candidate and every decision rather than only the accepted row. This makes
false-positive filters debuggable and lets later policy changes rebuild the dataset.


In [ ]:
from dataclasses import dataclass, asdict
import hashlib, json, random, re
@dataclass
class Candidate:
    seed_id: str; generator: str; prompt_version: str; instruction: str; response: str
    verifier: str | None = None; accepted: bool | None = None; reasons: tuple[str, ...] = ()
def fingerprint(record):
    canonical = json.dumps(asdict(record), sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(canonical.encode()).hexdigest()
example = Candidate("math-17", "teacher@revision", "synth-v3", "Compute 17*23", "391")
print(fingerprint(example), asdict(example))


## 14.2 Seed distribution and generation

Start from an explicit capability taxonomy and sample seeds to cover it. Uniform task counts rarely
mean uniform difficulty, language, or token volume. Stratify by domain, skill, difficulty, format,
safety class, language, and source. Hold out evaluation templates and source documents before any
teacher sees them; otherwise paraphrasing can contaminate the benchmark.

Record teacher ID and revision, tokenizer/template, system and generation prompts, sampling parameters,
seeds, tool calls, and generation time. Generate multiple candidates when selection is meaningful, but
account for correlated samples. Temperature increases surface diversity without guaranteeing semantic
diversity. Prompt mutation, multiple teachers, procedural generators, and retrieval-grounded creation
can cover different modes. Respect source and teacher licenses and document whether outputs may be used
for training or redistribution.


In [ ]:
taxonomy = {
    "arithmetic": ["integer multiplication", "fractions", "units"],
    "coding": ["implementation", "debugging", "tests"],
    "instruction": ["JSON schema", "constraints", "abstention"],
}
rng = random.Random(42)
generation_plan = [(domain, rng.choice(skills)) for domain, skills in taxonomy.items() for _ in range(3)]
print(generation_plan)


## 14.3 Verification beats self-confidence

Prefer independent, deterministic validators: execute tests in a sandbox, compare normalized exact
answers, validate JSON Schema, type-check programs, recompute arithmetic, confirm citations against
retrieved evidence, or run simulators. A second model can judge subjective properties but is another
noisy measurement instrument; calibrate it against blinded human labels and randomize presentation.
Never ask the generator to be the only judge of its own work.

Compose gates rather than one opaque score. Syntax, correctness, relevance, safety, and novelty have
different failure costs. Store per-gate outcomes and allow `unknown` instead of forcing every example
into pass/fail. Sample accepted and rejected rows for human audit; false acceptance poisons training,
while false rejection silently removes difficult or minority cases.


In [ ]:
def verify_integer(candidate, expected):
    match = re.fullmatch(r"\s*[-+]?\d+\s*", candidate.response)
    reasons = []
    if not match: reasons.append("not_integer")
    elif int(candidate.response) != expected: reasons.append("wrong_answer")
    candidate.verifier = "integer_exact_v1"
    candidate.accepted = not reasons; candidate.reasons = tuple(reasons)
    return candidate
rows = [Candidate("a", "teacher@rev", "v1", "17*23", answer) for answer in ["391", "390", "391 because..."]]
print([asdict(verify_integer(row, 391)) for row in rows])


## 14.4 Deduplication, leakage, and diversity

Exact hashes catch identical normalized text. Near-duplicate detection may use n-gram MinHash,
locality-sensitive hashing, embeddings, syntax trees, or task-specific canonicalization. Deduplicate
before splitting and compare candidates against evaluation corpora. Semantic similarity alone can
over-remove legitimate recurring forms or under-detect answer-preserving paraphrases, so inspect
thresholds by slice.

Diversity includes task semantics, reasoning strategy, response length, lexical style, language, and
error mode—not just unique strings. Measure source/teacher concentration and n-gram overlap, cluster
embeddings, and compare length/token distributions. Balance by effective tokens and downstream utility.
A difficulty curriculum may progress from verified simple examples to harder tasks, but “teacher wrote
more tokens” is not a difficulty metric.


In [ ]:
def normalize(text): return " ".join(text.lower().split())
texts = ["Return JSON only.", " return   json ONLY. ", "Explain JSON schemas."]
groups = {}
for text in texts: groups.setdefault(hashlib.sha256(normalize(text).encode()).hexdigest()[:8], []).append(text)
print(groups)


## 14.5 Mixtures, experiments, and release

Synthetic data should earn its place through ablation. Train matched runs with human-only data, each
synthetic source, and mixtures while holding tokens or compute constant. Evaluate target gains,
general capability retention, calibration, safety, and style artifacts. More accepted rows may reduce
quality if they dominate scarce high-quality demonstrations.

Publish a dataset card with purpose, schema, seed sources, generation and verification code revisions,
model revisions, licenses, counts at every gate, known errors, audits, demographics/languages, duplicate
policy, contamination tests, and intended uses. Version immutable shards and a manifest of hashes.
Do not place credentials, private prompts, personal data, or proprietary source passages in released
artifacts. The resulting dataset remains evidence with uncertainty—not ground truth merely because a
verifier assigned `1.0`.


## 14.6 Rejection sampling and verifier bias

Synthetic examples are often generated in excess and filtered by rules, models, or execution. Record the full funnel: attempts, parse success, verifier pass rate, deduplication, and final slice distribution. A verifier selects what it can recognize, which can narrow style and reward shortcuts. Measure false accepts and false rejects on human-reviewed samples, retain rejected examples for audits, and compare training with unfiltered and filtered subsets. Do not let the generator grade itself without independent checks.


In [ ]:
records=[{"parsed":1,"verified":1,"domain":"math"},{"parsed":1,"verified":0,"domain":"math"},{"parsed":0,"verified":0,"domain":"code"},{"parsed":1,"verified":1,"domain":"code"}]
print("parse rate",sum(r["parsed"] for r in records)/len(records),"accept rate",sum(r["verified"] for r in records)/len(records))


## 14.7 Provenance-aware synthetic records

A useful synthetic record contains prompt, response, generator revision, exact template and sampling configuration, seed, verifier versions and scores, transformation history, source references, license constraints, and content hashes. Separate generation from selection so filters can be rerun. Detect near duplicates against train and evaluation sets before training. Synthetic data can amplify teacher errors and reduce diversity even when every example looks polished; evaluate factuality, calibration, style entropy, safety, and downstream gains against a real-data baseline.


In [ ]:
import hashlib,json
record={"prompt":"Solve 2x=6","response":"x=3","generator":"model@commit","seed":7,"verifier":{"equation":True}}
canonical=json.dumps(record,sort_keys=True,separators=(",",":")); record["sha256"]=hashlib.sha256(canonical.encode()).hexdigest(); print(record)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Self-Instruct](https://arxiv.org/abs/2212.10560)
- [Textbooks Are All You Need](https://arxiv.org/abs/2306.11644)


## Exercises

    1. Generate a procedural arithmetic dataset and demonstrate independent verification and deduplication.
2. Design an audit that estimates false acceptance with a confidence interval.
3. Run a data-mixture ablation and report quality per training token, not only final score.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
